# Diabetes Deep Learning Assignment
## 1. Classification: Outcome 예측
## 2. Regression: Outcome 제거 후 BMI 예측


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score
)

tf.random.set_seed(42)
np.random.seed(42)


In [3]:
path = "/content/drive/MyDrive/Colab Notebooks/csv_file/diabetes.csv"
df = pd.read_csv(path)

print(df.head())
print(df.shape)
print(df.columns)


   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
(768, 9)
Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')


## 1. Classification
`Outcome`을 정답(y)으로 두고 나머지 8개 열을 입력(X)으로 사용한다.


In [4]:
X = df.drop("Outcome", axis=1).values
y = df["Outcome"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler_cls = StandardScaler()
X_train = scaler_cls.fit_transform(X_train)
X_test = scaler_cls.transform(X_test)

print(X_train.shape, X_test.shape)


(614, 8) (154, 8)


In [5]:
model_cls = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model_cls.compile(
    optimizer="sgd",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_cls.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 833 (3.25 KB)

 Trainable params: 833 (3.25 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
history_cls = model_cls.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


Epoch 1/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.4399 - loss: 0.7116 - val_accuracy: 0.4553 - val_loss: 0.6877
Epoch 2/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4847 - loss: 0.6951 - val_accuracy: 0.5285 - val_loss: 0.6733
Epoch 3/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5316 - loss: 0.6813 - val_accuracy: 0.6260 - val_loss: 0.6607
Epoch 4/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6110 - loss: 0.6694 - val_accuracy: 0.6341 - val_loss: 0.6498
Epoch 5/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6619 - loss: 0.6591 - val_accuracy: 0.6667 - val_loss: 0.6402
Epoch 6/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6782 - loss: 0.6499 - val_accuracy: 0.7073 - val_loss: 0.6315
Epoch 7/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7108 - loss: 0.6416 - val_accuracy: 0.6911 - val_loss: 0.6234
Epoch 8/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7189 - loss: 0.6341 - val_accuracy: 0.7236 - val_loss

In [7]:
y_prob = model_cls.predict(X_test)
y_pred = (y_prob >= 0.5).astype(int).flatten()

print("Accuracy =", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred))


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Accuracy = 0.6883116883116883

Confusion Matrix
[[82 18]
 [30 24]]

Classification Report
              precision    recall  f1-score   support

           0       0.73      0.82      0.77       100
           1       0.57      0.44      0.50        54

    accuracy                           0.69       154
   macro avg       0.65      0.63      0.64       154
weighted avg       0.68      0.69      0.68       154



## 2. Regression
`Outcome` 열을 제거한 뒤 `BMI`를 정답(y)으로 두고 나머지 7개 열로 BMI를 예측한다.


In [8]:
df_reg = df.drop("Outcome", axis=1)

X_reg = df_reg.drop("BMI", axis=1).values
y_reg = df_reg["BMI"].values

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg,
    test_size=0.2,
    random_state=42
)

scaler_reg = StandardScaler()
X_train_r = scaler_reg.fit_transform(X_train_r)
X_test_r = scaler_reg.transform(X_test_r)

print(X_train_r.shape, X_test_r.shape)


(614, 7) (154, 7)


In [9]:
model_reg = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(7,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1)
])

model_reg.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model_reg.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 32)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 801 (3.13 KB)

 Trainable params: 801 (3.13 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history_reg = model_reg.fit(
    X_train_r, y_train_r,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


Epoch 1/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 949.9326 - mae: 29.8052 - val_loss: 605.8731 - val_mae: 23.4884
Epoch 2/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 210.5114 - mae: 11.5381 - val_loss: 75.2617 - val_mae: 6.6555
Epoch 3/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 61.2034 - mae: 5.6628 - val_loss: 56.5395 - val_mae: 5.8470
Epoch 4/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 51.7782 - mae: 5.2219 - val_loss: 53.0164 - val_mae: 5.7558
Epoch 5/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 48.4844 - mae: 5.0673 - val_loss: 49.2717 - val_mae: 5.5936
Epoch 6/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 46.2312 - mae: 4.9325 - val_loss: 49.7495 - val_mae: 5.6130
Epoch 7/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 44.9885 - mae: 4.8739 - val_loss: 48.8865 - val_mae: 5.5775
Epoch 8/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 43.8350 - mae: 4.8089 - val_loss: 49.1289 - val_mae: 5.6006
Epoch 9/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

In [11]:
y_pred_r = model_reg.predict(X_test_r).flatten()

print("MSE =", mean_squared_error(y_test_r, y_pred_r))
print("MAE =", mean_absolute_error(y_test_r, y_pred_r))
print("R2 =", r2_score(y_test_r, y_pred_r))

print("\n예측값 일부 :", y_pred_r[:10])
print("실제값 일부 :", y_test_r[:10])


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
MSE = 68.65114189420953
MAE = 5.700578954622343
R2 = 0.034244745297409396

예측값 일부 : [35.040546 32.10851  29.127932 33.452435 31.626686 34.71136  24.993876
 26.111948 31.149126 27.513044]
실제값 일부 : [34.  35.7 30.8 24.6 29.9 37.7 20.4 33.8 31.3 33.7]
